# Portfolio Optimization AI - Data Exploration

This notebook explores the portfolio optimization system and demonstrates its capabilities.

## Overview

The system combines:
- **Machine Learning**: RandomForest and XGBoost for return prediction
- **Modern Portfolio Theory**: Markowitz mean-variance optimization
- **Risk Analysis**: Comprehensive risk metrics
- **Feature Engineering**: 20+ technical indicators

## 1. Setup and Imports

In [ ]:
# Import necessary libraries
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Add src directory to path
sys.path.append('../src')

# Import portfolio optimization modules
from data_loader import DataLoader
from feature_engineering import FeatureEngineer
from return_predictor import ReturnPredictor
from portfolio_optimizer import PortfolioOptimizer
from risk_metrics import RiskMetrics
from pipeline import PortfolioOptimizationPipeline

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ All imports successful!")

## 2. Data Loading and Exploration

In [ ]:
# Define tickers for analysis
tickers = ["AAPL", "MSFT", "GOOG", "AMZN", "META", "TSLA"]
print(f"📊 Analyzing {len(tickers)} stocks: {', '.join(tickers)}")

# Initialize data loader
data_loader = DataLoader()

# Download historical data
print("\n📥 Downloading historical data...")
price_data = data_loader.download_stock_data(tickers, period="5y")

print(f"\n📈 Data shape: {price_data.shape}")
print(f"📅 Date range: {price_data.index[0].date()} to {price_data.index[-1].date()}")
print(f"💰 Tickers: {list(price_data.columns)}")

# Display first few rows
print("\n📋 Sample data:")
display(price_data.head())

In [ ]:
# Calculate and visualize returns
returns = data_loader.get_returns(method='simple')

# Plot price evolution
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Stock Prices", "Daily Returns"),
    vertical_spacing=0.1
)

# Add price lines
for ticker in price_data.columns:
    fig.add_trace(
        go.Scatter(
            x=price_data.index,
            y=price_data[ticker],
            name=ticker,
            mode='lines'
        ),
        row=1, col=1
    )

# Add return lines
for ticker in returns.columns:
    fig.add_trace(
        go.Scatter(
            x=returns.index,
            y=returns[ticker],
            name=f"{ticker} Returns",
            mode='lines',
            opacity=0.7
        ),
        row=2, col=1
    )

fig.update_layout(
    title="Stock Prices and Returns",
    height=800,
    showlegend=True
)

fig.show()

In [ ]:
# Calculate and display return statistics
return_stats = pd.DataFrame({
    'Mean Annual Return': returns.mean() * 252,
    'Annual Volatility': returns.std() * np.sqrt(252),
    'Sharpe Ratio': (returns.mean() * 252) / (returns.std() * np.sqrt(252)),
    'Skewness': returns.skew(),
    'Kurtosis': returns.kurtosis()
})

print("📊 Return Statistics:")
display(return_stats.style.format({
    'Mean Annual Return': '{:.2%}',
    'Annual Volatility': '{:.2%}',
    'Sharpe Ratio': '{:.3f}',
    'Skewness': '{:.3f}',
    'Kurtosis': '{:.3f}'
}))

# Correlation matrix
correlation_matrix = returns.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm',
    center=0,
    fmt='.3f'
)
plt.title('Correlation Matrix of Returns')
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
# Initialize feature engineer
feature_engineer = FeatureEngineer()

# Create comprehensive feature matrix
print("🔧 Creating feature matrix...")
features = feature_engineer.create_feature_matrix(price_data)

print(f"\n📊 Feature matrix shape: {features.shape}")
print(f"📈 Total features created: {len(features.columns)}")

# Get feature summary
feature_summary = feature_engineer.get_feature_summary()
print("\n📋 Feature categories:")
for category, count in feature_summary['feature_categories'].items():
    print(f"  {category}: {count} features")

# Display sample features
print("\n📋 Sample features:")
display(features.head())

In [ ]:
# Visualize some key features for AAPL
aapl_features = [col for col in features.columns if 'AAPL' in col]
aapl_df = features[aapl_features]

# Select a subset of features for visualization
key_features = [
    'AAPL_simple_return',
    'AAPL_volatility_20d',
    'AAPL_rsi_14',
    'AAPL_momentum_5d',
    'AAPL_macd'
]

# Plot key features
fig = make_subplots(
    rows=len(key_features), cols=1,
    subplot_titles=key_features,
    vertical_spacing=0.02
)

for i, feature in enumerate(key_features):
    if feature in aapl_df.columns:
        fig.add_trace(
            go.Scatter(
                x=aapl_df.index,
                y=aapl_df[feature],
                mode='lines',
                name=feature,
                showlegend=False
            ),
            row=i+1, col=1
        )

fig.update_layout(
    title="Key Technical Indicators for AAPL",
    height=1000,
    showlegend=False
)

fig.show()

## 4. Machine Learning Return Prediction

In [ ]:
# Initialize return predictor
predictor = ReturnPredictor()

# Train models for each ticker
model_results = {}

print("🤖 Training ML models for return prediction...")

for ticker in tickers:
    print(f"\n📈 Training models for {ticker}...")
    
    try:
        # Get features and target for this ticker
        X, y = feature_engineer.get_feature_importance_data(ticker)
        
        if len(X) < 100:
            print(f"  ⚠️  Insufficient data for {ticker}, skipping...")
            continue
        
        # Train Random Forest
        rf_results = predictor.train_random_forest(X, y, f"rf_{ticker}")
        
        # Train XGBoost
        xgb_results = predictor.train_xgboost(X, y, f"xgb_{ticker}")
        
        # Get best model
        best_model_name, best_performance = predictor.get_best_model()
        
        model_results[ticker] = {
            'best_model': best_model_name,
            'test_r2': best_performance['test']['r2'],
            'test_rmse': best_performance['test']['rmse']
        }
        
        print(f"  ✅ Best model: {best_model_name}")
        print(f"  📊 Test R²: {best_performance['test']['r2']:.4f}")
        print(f"  📊 Test RMSE: {best_performance['test']['rmse']:.4f}")
        
    except Exception as e:
        print(f"  ❌ Error training models for {ticker}: {str(e)}")
        continue

In [ ]:
# Display model performance summary
if model_results:
    performance_df = pd.DataFrame(model_results).T
    
    print("📊 Model Performance Summary:")
    display(performance_df.style.format({
        'test_r2': '{:.4f}',
        'test_rmse': '{:.4f}'
    }))
    
    # Plot performance comparison
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=performance_df.index,
        y=performance_df['test_r2'],
        name='Test R²',
        marker_color='lightblue'
    ))
    
    fig.update_layout(
        title='Model Performance (Test R²) by Ticker',
        xaxis_title='Ticker',
        yaxis_title='R² Score',
        showlegend=False
    )
    
    fig.show()
else:
    print("❌ No models were successfully trained")

## 5. Portfolio Optimization

In [ ]:
# Get expected returns from ML models
expected_returns = {}

print("🎯 Getting expected returns from ML models...")

for ticker in tickers:
    try:
        # Get features for this ticker
        ticker_features = {col: features[col] for col in features.columns if ticker in col}
        ticker_features_df = pd.DataFrame(ticker_features)
        
        if ticker in model_results:
            best_model = model_results[ticker]['best_model']
            predicted_return = predictor.predict_returns(
                ticker_features_df.iloc[-1:].copy(), best_model
            )
            expected_returns[ticker] = float(predicted_return[0]) * 252  # Annualize
            print(f"  📈 {ticker}: {expected_returns[ticker]:.2%} (using {best_model})")
        else:
            # Fallback to historical mean
            hist_return = returns[ticker].mean() * 252
            expected_returns[ticker] = hist_return
            print(f"  📈 {ticker}: {expected_returns[ticker]:.2%} (historical mean)")
    except Exception as e:
        print(f"  ❌ Error for {ticker}: {str(e)}")
        expected_returns[ticker] = 0.0

# Calculate covariance matrix
covariance_matrix = returns.cov() * 252  # Annualized

print(f"\n📊 Expected Returns:")
for ticker, ret in expected_returns.items():
    print(f"  {ticker}: {ret:.2%}")

In [ ]:
# Initialize portfolio optimizer
optimizer = PortfolioOptimizer(risk_free_rate=0.02)
optimizer.set_inputs(expected_returns, covariance_matrix)

# Run different optimization strategies
print("🎯 Running portfolio optimization...")

# Maximum Sharpe ratio
max_sharpe_result = optimizer.optimize_max_sharpe()
print(f"\n📈 Maximum Sharpe Ratio Portfolio:")
print(f"  Expected Return: {max_sharpe_result.expected_return:.2%}")
print(f"  Volatility: {max_sharpe_result.volatility:.2%}")
print(f"  Sharpe Ratio: {max_sharpe_result.sharpe_ratio:.3f}")

# Minimum volatility
min_vol_result = optimizer.optimize_min_volatility()
print(f"\n🛡️  Minimum Volatility Portfolio:")
print(f"  Expected Return: {min_vol_result.expected_return:.2%}")
print(f"  Volatility: {min_vol_result.volatility:.2%}")
print(f"  Sharpe Ratio: {min_vol_result.sharpe_ratio:.3f}")

# Equal weight
equal_weight_result = optimizer.optimize_equal_weight()
print(f"\n⚖️  Equal Weight Portfolio:")
print(f"  Expected Return: {equal_weight_result.expected_return:.2%}")
print(f"  Volatility: {equal_weight_result.volatility:.2%}")
print(f"  Sharpe Ratio: {equal_weight_result.sharpe_ratio:.3f}")

In [ ]:
# Compare portfolio allocations
comparison_data = []

for ticker in tickers:
    comparison_data.append({
        'Ticker': ticker,
        'Max Sharpe': max_sharpe_result.weights.get(ticker, 0),
        'Min Volatility': min_vol_result.weights.get(ticker, 0),
        'Equal Weight': equal_weight_result.weights.get(ticker, 0)
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.set_index('Ticker')

# Plot comparison
fig = go.Figure()

for strategy in ['Max Sharpe', 'Min Volatility', 'Equal Weight']:
    fig.add_trace(go.Bar(
        x=comparison_df.index,
        y=comparison_df[strategy],
        name=strategy,
        opacity=0.8
    ))

fig.update_layout(
    title='Portfolio Allocation Comparison',
    xaxis_title='Ticker',
    yaxis_title='Weight',
    barmode='group',
    height=500
)

fig.show()

# Display allocation table
print("📊 Portfolio Allocation Comparison:")
display(comparison_df.style.format({
    'Max Sharpe': '{:.2%}',
    'Min Volatility': '{:.2%}',
    'Equal Weight': '{:.2%}'
}))

## 6. Risk Analysis

In [ ]:
# Calculate portfolio returns for risk analysis
portfolio_returns = pd.Series(0.0, index=returns.index)

for ticker, weight in max_sharpe_result.weights.items():
    if ticker in returns.columns:
        portfolio_returns += returns[ticker] * weight

# Calculate portfolio values
portfolio_values = (1 + portfolio_returns).cumprod()

# Initialize risk calculator
risk_calculator = RiskMetrics(risk_free_rate=0.02)

# Comprehensive risk analysis
risk_analysis = risk_calculator.comprehensive_risk_analysis(
    portfolio_returns, portfolio_values
)

print("⚠️  Risk Analysis for Max Sharpe Portfolio:")
print(f"  Sharpe Ratio: {risk_analysis['sharpe_ratio']:.3f}")
print(f"  Sortino Ratio: {risk_analysis['sortino_ratio']:.3f}")
print(f"  Maximum Drawdown: {risk_analysis['drawdown_analysis']['max_drawdown']:.2%}")
print(f"  VaR (95%): {risk_analysis['var_95%']['var_5%']:.2%}")
print(f"  Expected Shortfall (95%): {risk_analysis['var_95%']['expected_shortfall_5%']:.2%}")
print(f"  Calmar Ratio: {risk_analysis['calmar_ratio']:.3f}")
print(f"  Omega Ratio: {risk_analysis['omega_ratio']:.3f}")

In [ ]:
# Plot drawdown analysis
drawdown_data = risk_analysis['drawdown_analysis']

# Calculate cumulative returns and drawdown
cumulative_returns = (1 + portfolio_returns).cumprod()
running_max = cumulative_returns.expanding().max()
drawdown = (cumulative_returns - running_max) / running_max

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Portfolio Value", "Drawdown"),
    vertical_spacing=0.1
)

# Portfolio value
fig.add_trace(
    go.Scatter(
        x=portfolio_values.index,
        y=portfolio_values,
        mode='lines',
        name='Portfolio Value',
        line=dict(color='blue')
    ),
    row=1, col=1
)

# Drawdown
fig.add_trace(
    go.Scatter(
        x=drawdown.index,
        y=drawdown,
        mode='lines',
        name='Drawdown',
        fill='tonexty',
        line=dict(color='red')
    ),
    row=2, col=1
)

# Add zero line for drawdown
fig.add_hline(y=0, line_dash="dash", line_color="gray", row=2, col=1)

fig.update_layout(
    title="Portfolio Performance and Drawdown Analysis",
    height=600,
    showlegend=True
)

fig.show()

## 7. Efficient Frontier and Monte Carlo Simulation

In [ ]:
# Generate efficient frontier
print("📈 Generating efficient frontier...")
efficient_frontier = optimizer.generate_efficient_frontier(num_portfolios=50)

# Monte Carlo simulation
print("🎲 Running Monte Carlo simulation...")
monte_carlo_results = optimizer.monte_carlo_simulation(num_portfolios=1000)

# Plot efficient frontier and Monte Carlo results
fig = go.Figure()

# Monte Carlo portfolios
fig.add_trace(go.Scatter(
    x=monte_carlo_results['volatility'],
    y=monte_carlo_results['return'],
    mode='markers',
    marker=dict(
        color=monte_carlo_results['sharpe_ratio'],
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Sharpe Ratio")
    ),
    name='Monte Carlo Portfolios',
    opacity=0.6
))

# Efficient frontier
if len(efficient_frontier) > 0:
    fig.add_trace(go.Scatter(
        x=efficient_frontier['volatility'],
        y=efficient_frontier['return'],
        mode='lines+markers',
        name='Efficient Frontier',
        line=dict(color='red', width=3),
        marker=dict(size=6)
    ))

# Optimal portfolios
fig.add_trace(go.Scatter(
    x=[max_sharpe_result.volatility],
    y=[max_sharpe_result.expected_return],
    mode='markers',
    marker=dict(
        symbol='star',
        size=15,
        color='gold'
    ),
    name='Max Sharpe Portfolio',
    text=[f"Sharpe: {max_sharpe_result.sharpe_ratio:.3f}"],
    textposition="top center"
))

fig.add_trace(go.Scatter(
    x=[min_vol_result.volatility],
    y=[min_vol_result.expected_return],
    mode='markers',
    marker=dict(
        symbol='diamond',
        size=12,
        color='orange'
    ),
    name='Min Volatility Portfolio'
))

fig.update_layout(
    title="Efficient Frontier and Portfolio Optimization",
    xaxis_title="Volatility (Risk)",
    yaxis_title="Expected Return",
    height=600,
    showlegend=True
)

fig.show()

# Display Monte Carlo statistics
mc_stats = {
    'Max Sharpe Ratio': monte_carlo_results['sharpe_ratio'].max(),
    'Min Volatility': monte_carlo_results['volatility'].min(),
    'Max Return': monte_carlo_results['return'].max(),
    'Min Return': monte_carlo_results['return'].min()
}

print("\n📊 Monte Carlo Simulation Statistics:")
for metric, value in mc_stats.items():
    print(f"  {metric}: {value:.4f}")

## 8. End-to-End Pipeline Demo

In [ ]:
# Run the complete pipeline
print("🚀 Running complete portfolio optimization pipeline...")

# Initialize pipeline
pipeline = PortfolioOptimizationPipeline()

# Run optimization
pipeline_results = pipeline.run(
    tickers=tickers,
    optimization_method="max_sharpe"
)

# Display results
print("\n✅ Pipeline completed successfully!")
print(f"📊 Expected Return: {pipeline_results['optimization_result']['expected_return']:.2%}")
print(f"📊 Volatility: {pipeline_results['optimization_result']['volatility']:.2%}")
print(f"📊 Sharpe Ratio: {pipeline_results['optimization_result']['sharpe_ratio']:.3f}")

print("\n🎯 Optimal Portfolio Weights:")
for ticker, weight in pipeline_results['optimization_result']['weights'].items():
    print(f"  {ticker}: {weight:.2%}")

# Display risk metrics
if 'risk_metrics' in pipeline_results:
    risk_metrics = pipeline_results['risk_metrics']
    print("\n⚠️  Key Risk Metrics:")
    print(f"  Sortino Ratio: {risk_metrics.get('sortino_ratio', 'N/A')}")
    print(f"  Max Drawdown: {risk_metrics.get('drawdown_analysis', {}).get('max_drawdown', 'N/A'):.2%}")
    print(f"  VaR (95%): {risk_metrics.get('var_95%', {}).get('var_5%', 'N/A'):.2%}")

## 9. Summary and Insights

In [ ]:
# Create summary dashboard
summary_data = {
    'Metric': [
        'Expected Return',
        'Volatility',
        'Sharpe Ratio',
        'Sortino Ratio',
        'Max Drawdown',
        'VaR (95%)',
        'Number of Assets',
        'Best ML Model (Avg R²)'
    ],
    'Value': [
        f"{pipeline_results['optimization_result']['expected_return']:.2%}",
        f"{pipeline_results['optimization_result']['volatility']:.2%}",
        f"{pipeline_results['optimization_result']['sharpe_ratio']:.3f}",
        f"{pipeline_results['risk_metrics'].get('sortino_ratio', 0):.3f}",
        f"{pipeline_results['risk_metrics'].get('drawdown_analysis', {}).get('max_drawdown', 0):.2%}",
        f"{pipeline_results['risk_metrics'].get('var_95%', {}).get('var_5%', 0):.2%}",
        len(pipeline_results['optimization_result']['weights']),
        f"{np.mean([r['test_r2'] for r in model_results.values()]):.3f}" if model_results else 'N/A'
    ]
}

summary_df = pd.DataFrame(summary_data)

print("📊 Portfolio Optimization Summary:")
display(summary_df)

# Key insights
print("\n💡 Key Insights:")
print(f"  • The optimized portfolio achieves a Sharpe ratio of {pipeline_results['optimization_result']['sharpe_ratio']:.3f}")
print(f"  • Expected annual return of {pipeline_results['optimization_result']['expected_return']:.2%} with {pipeline_results['optimization_result']['volatility']:.2%} volatility")
print(f"  • Maximum drawdown is limited to {pipeline_results['risk_metrics'].get('drawdown_analysis', {}).get('max_drawdown', 0):.2%}")
print(f"  • ML models achieved an average R² of {np.mean([r['test_r2'] for r in model_results.values()]):.3f} in return prediction")
print(f"  • Portfolio is diversified across {len(pipeline_results['optimization_result']['weights'])} assets")

if 'insights' in pipeline_results and 'concentration' in pipeline_results['insights']:
    concentration = pipeline_results['insights']['concentration']
    print(f"  • Herfindahl-Hirschman Index: {concentration.get('herfindahl_index', 0):.3f} (lower = more diversified)")
    print(f"  • Maximum single asset weight: {concentration.get('max_weight', 0):.2%}")

## 10. Next Steps

### 🚀 Production Deployment
1. **API Service**: `uvicorn api.server:app --host 0.0.0.0 --port 8000`
2. **Dashboard**: `streamlit run dashboard/app.py --server.port 8501`
3. **Docker**: `cd deployment && ./run.sh`

### 🔧 Customization Options
- **Add more assets**: Bonds, commodities, cryptocurrencies
- **Enhanced models**: LSTM neural networks, ensemble methods
- **Alternative optimization**: Risk parity, factor models
- **Backtesting**: Historical performance analysis

### 📈 Extensions
- **Real-time data**: Live market data integration
- **Rebalancing**: Automated portfolio rebalancing
- **Risk management**: Dynamic risk limits and stop-loss
- **Performance attribution**: Factor-based performance analysis

---

**🎉 Congratulations!** You've successfully explored the Portfolio Optimization AI system. The notebook demonstrated:

✅ **Data Loading**: Historical stock price data from Yahoo Finance
✅ **Feature Engineering**: 20+ technical indicators
✅ **ML Prediction**: RandomForest and XGBoost for return forecasting
✅ **Portfolio Optimization**: Modern Portfolio Theory implementation
✅ **Risk Analysis**: Comprehensive risk metrics
✅ **Visualization**: Interactive charts and insights

Ready to deploy? Check the README.md for detailed instructions!